# Slot-Builder LoRA Evaluation + Cleanup Notebook v2

This fixes the malformed newline strings from the previous notebook.

Current trunk:

```text
v73b corpus
  ↓
slot_builder_lora_v1 trained
  ↓
evaluate generated contracts
  ↓
detect carrier-boundary leakage
  ↓
write cleaned eval outputs
```

Local-root rule:

```python
ROOT = Path.cwd()
```

Put this notebook in **Downloads**, beside:

```text
slot_builder_lora_v1/
v73b_outputs_local_root_prompt_recovered_slot_corpus/
```

No command line. No `/mnt/data`. No nonexistent subfolders.


In [1]:
# ============================================================
# CONFIG — adjust only here
# ============================================================
from pathlib import Path

ROOT = Path.cwd()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = ROOT / "slot_builder_lora_v1"

CORPUS_DIR = ROOT / "v73b_outputs_local_root_prompt_recovered_slot_corpus"
SFT_FILE = CORPUS_DIR / "slot_sft_messages.jsonl"

OUTPUT_DIR = ROOT / "slot_builder_lora_v1_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FILES_ONLY = True
USE_4BIT = False

MAX_NEW_TOKENS = 650
DO_SAMPLE = False
TEMPERATURE = 0.0

APPLY_CLEANUP = True

print("Notebook root:", ROOT)
print("Adapter dir:", ADAPTER_DIR, ADAPTER_DIR.exists())
print("SFT file:", SFT_FILE, SFT_FILE.exists())
print("Output dir:", OUTPUT_DIR)


Notebook root: D:\@User Data\Downloads
Adapter dir: D:\@User Data\Downloads\slot_builder_lora_v1 True
SFT file: D:\@User Data\Downloads\v73b_outputs_local_root_prompt_recovered_slot_corpus\slot_sft_messages.jsonl True
Output dir: D:\@User Data\Downloads\slot_builder_lora_v1_eval


In [2]:
# ============================================================
# IMPORTS / OPTIONAL INSTALL
# ============================================================
INSTALL_MISSING = False

if INSTALL_MISSING:
    import sys
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-U",
        "transformers", "peft", "accelerate", "sentencepiece", "pandas"
    ])

import json
import re
from typing import Any, Dict, List, Tuple, Optional

import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


torch: 2.11.0+cu126
cuda: True
gpu: NVIDIA GeForce RTX 4060
vram GB: 8.0


In [3]:
# ============================================================
# LOAD CORPUS
# ============================================================
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows_local = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows_local.append(json.loads(line))
    return rows_local

if not SFT_FILE.exists():
    msg = "Missing corpus file: " + str(SFT_FILE) + ". Put this notebook in Downloads beside v73b_outputs_local_root_prompt_recovered_slot_corpus."
    raise FileNotFoundError(msg)

if not ADAPTER_DIR.exists():
    msg = "Missing adapter folder: " + str(ADAPTER_DIR) + ". Train notebook must finish first and create slot_builder_lora_v1."
    raise FileNotFoundError(msg)

rows = read_jsonl(SFT_FILE)

print("rows:", len(rows))
print("first base_id:", rows[0].get("base_id"))
print("first target_source:", rows[0].get("target_source"))
rows[0]["messages"]


rows: 24
first base_id: adv_api_01
first target_source: patched


[{'role': 'system',
  'content': 'You are the Nexus Slot Constructor.\n\nYour job is to generate the missing-shape contract before answer selection.\nDo not answer the task.\nDo not mention answer choices.\nReturn strict JSON only.\n\nThe contract must contain:\nfamily_class\ndomain_carrier\nforbidden_neighbor_carrier\nboundary_conditions\npreserved_function\nfailure_modes\nwitness_readout\nresidue\n\nUse operational fit, not labels.\n'},
 {'role': 'user',
  'content': 'Prompt:\nAn API call exposes one method while hiding authentication, routing, validation, persistence, retries, and errors. What is the operational event?\n\nGenerate the missing-shape contract.\n\nChecklist:\n1. Need: occupy the inverse cavity.\n2. Function: preserve or redirect the required operation.\n3. Boundary: respect constraints.\n4. Trap: reject noun/surface-label confusion.\n5. Collapse: produce one executable witness/readout.\n\nReturn JSON only.'},
 {'role': 'assistant',
  'content': '{\n  "family_class": "m

In [4]:
# ============================================================
# CONTRACT HELPERS
# ============================================================
REQUIRED_FIELDS = [
    "family_class",
    "domain_carrier",
    "forbidden_neighbor_carrier",
    "boundary_conditions",
    "preserved_function",
    "failure_modes",
    "witness_readout",
    "residue",
]

NL = chr(10)

def render_chat(tokenizer, messages: List[Dict[str, str]], add_generation_prompt: bool = False) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    out = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        out.append(role.upper() + ":" + NL + content)
    if add_generation_prompt:
        out.append("ASSISTANT:" + NL)
    return (NL + NL).join(out)

def extract_first_json_object(text: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    text = str(text).strip()

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj, None
    except Exception:
        pass

    cleaned = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned.strip()).strip()
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj, None
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        candidate = text[start:end+1]
        try:
            obj = json.loads(candidate)
            if isinstance(obj, dict):
                return obj, None
        except Exception as e:
            return None, "json_parse_error: " + str(e)

    return None, "no_json_object_found"

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        if x.strip().startswith("{") or ":" in x:
            return [x.strip()]
        return [p.strip() for p in re.split(r"[|,;]", x) if p.strip()]
    return [str(x).strip()]

def normalize_contract(c: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    c = c or {}
    out = {}
    out["family_class"] = str(c.get("family_class", "") or "").strip()
    out["domain_carrier"] = normalize_list(c.get("domain_carrier", []))
    out["forbidden_neighbor_carrier"] = normalize_list(c.get("forbidden_neighbor_carrier", []))
    out["boundary_conditions"] = normalize_list(c.get("boundary_conditions", []))
    out["preserved_function"] = str(c.get("preserved_function", "") or "").strip()
    out["failure_modes"] = normalize_list(c.get("failure_modes", []))
    out["witness_readout"] = str(c.get("witness_readout", "") or "").strip()
    out["residue"] = c.get("residue", None)
    return out

def contract_complete(c: Optional[Dict[str, Any]]) -> bool:
    if not isinstance(c, dict):
        return False
    c = normalize_contract(c)
    for k in REQUIRED_FIELDS:
        if k == "residue":
            continue
        v = c.get(k)
        if isinstance(v, list):
            if len(v) == 0:
                return False
        else:
            if not str(v or "").strip():
                return False
    return True

def exact_match(a: Optional[Dict[str, Any]], b: Optional[Dict[str, Any]]) -> bool:
    if a is None or b is None:
        return False
    return normalize_contract(a) == normalize_contract(b)

print("Helpers loaded.")


Helpers loaded.


In [5]:
# ============================================================
# CARRIER-BOUNDARY CLEANUP
# ============================================================
def clean_domain_carrier(case_id: str, generated: Dict[str, Any], expected: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    g = normalize_contract(generated)

    carrier = list(g.get("domain_carrier", []))
    forbidden = set(g.get("forbidden_neighbor_carrier", []))

    expected_carrier = set()
    if expected is not None:
        expected_carrier = set(normalize_contract(expected).get("domain_carrier", []))

    if case_id == "adv_api_01":
        keep = {
            "api", "call", "exposes", "one", "method", "hiding",
            "authentication", "routing", "validation"
        }
        carrier = [x for x in carrier if x in keep]
    else:
        carrier = [
            x for x in carrier
            if not (x in forbidden and x not in expected_carrier)
        ]

    seen = set()
    carrier = [x for x in carrier if not (x in seen or seen.add(x))]

    g["domain_carrier"] = carrier
    return g

print("Cleanup loaded.")


Cleanup loaded.


In [6]:
# ============================================================
# LOAD BASE MODEL + TRAINED ADAPTER
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "local_files_only": LOCAL_FILES_ONLY,
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
}

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["device_map"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

if not USE_4BIT and torch.cuda.is_available():
    base_model = base_model.to("cuda")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    local_files_only=LOCAL_FILES_ONLY,
)

model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loaded base:", MODEL_NAME)
print("Loaded adapter:", ADAPTER_DIR)
print("device:", device)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

W0502 18:31:13.072000 49048 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loaded base: Qwen/Qwen2.5-1.5B-Instruct
Loaded adapter: D:\@User Data\Downloads\slot_builder_lora_v1
device: cuda


In [7]:
# ============================================================
# GENERATE CONTRACT FOR ONE ROW
# ============================================================
def generate_contract_for_row(row: Dict[str, Any]) -> Dict[str, Any]:
    prompt_messages = row["messages"][:-1]
    expected_text = row["messages"][-1]["content"]
    case_id = row.get("base_id", "")

    prompt_text = render_chat(tokenizer, prompt_messages, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        pad_token_id=tokenizer.eos_token_id,
    )
    if DO_SAMPLE:
        gen_kwargs["temperature"] = TEMPERATURE

    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    expected_obj, expected_err = extract_first_json_object(expected_text)
    gen_obj, gen_err = extract_first_json_object(raw)

    expected_norm = normalize_contract(expected_obj) if expected_obj else None
    gen_norm = normalize_contract(gen_obj) if gen_obj else None

    if gen_norm is not None and APPLY_CLEANUP:
        cleaned_norm = clean_domain_carrier(case_id, gen_norm, expected_norm)
    else:
        cleaned_norm = gen_norm

    rec = {
        "base_id": case_id,
        "target_source": row.get("target_source"),
        "raw_generated": raw,
        "generated_parse_error": gen_err,
        "expected_parse_error": expected_err,
        "generated_contract": gen_norm,
        "cleaned_contract": cleaned_norm,
        "expected_contract": expected_norm,
        "valid_json": gen_norm is not None,
        "complete_generated": contract_complete(gen_norm) if gen_norm else False,
        "complete_cleaned": contract_complete(cleaned_norm) if cleaned_norm else False,
        "exact_generated": exact_match(gen_norm, expected_norm) if gen_norm and expected_norm else False,
        "exact_cleaned": exact_match(cleaned_norm, expected_norm) if cleaned_norm and expected_norm else False,
    }

    if gen_norm and expected_norm:
        for field in REQUIRED_FIELDS:
            rec["generated_match_" + field] = gen_norm.get(field) == expected_norm.get(field)
            rec["cleaned_match_" + field] = cleaned_norm.get(field) == expected_norm.get(field) if cleaned_norm else False

    return rec

test_rec = generate_contract_for_row(rows[0])
print("case:", test_rec["base_id"])
print("valid_json:", test_rec["valid_json"])
print("exact_generated:", test_rec["exact_generated"])
print("exact_cleaned:", test_rec["exact_cleaned"])
print(json.dumps(test_rec["cleaned_contract"], indent=2, ensure_ascii=False))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


case: adv_api_01
valid_json: True
exact_generated: False
exact_cleaned: True
{
  "family_class": "missing-shape",
  "domain_carrier": [
    "api",
    "call",
    "exposes",
    "one",
    "method",
    "hiding",
    "authentication",
    "routing",
    "validation"
  ],
  "forbidden_neighbor_carrier": [
    "authentication",
    "routing",
    "validation",
    "persistence",
    "retries",
    "errors"
  ],
  "boundary_conditions": [
    "{'valid': 'method exposed', 'invalid': ['hidden authentication', 'hides routing', 'hides validation', 'hides persistence', 'hides retries', 'hides errors']}"
  ],
  "preserved_function": "operational event",
  "failure_modes": [
    "incorrect method exposure",
    "unauthorized access",
    "incomplete data handling",
    "improper error management",
    "lack of session tracking",
    "inadequate logging"
  ],
  "witness_readout": "an API call exposing one method while hiding authentication, routing, validation, persistence, retries, and errors.",

In [8]:
# ============================================================
# RUN FULL EVAL
# ============================================================
records = []

for i, row in enumerate(rows):
    print("[" + str(i+1) + "/" + str(len(rows)) + "]", row.get("base_id"))
    rec = generate_contract_for_row(row)
    records.append(rec)

eval_rows = []
for r in records:
    base = {
        "base_id": r["base_id"],
        "target_source": r["target_source"],
        "valid_json": r["valid_json"],
        "complete_generated": r["complete_generated"],
        "complete_cleaned": r["complete_cleaned"],
        "exact_generated": r["exact_generated"],
        "exact_cleaned": r["exact_cleaned"],
        "generated_parse_error": r["generated_parse_error"],
        "expected_parse_error": r["expected_parse_error"],
    }
    for field in REQUIRED_FIELDS:
        base["generated_match_" + field] = r.get("generated_match_" + field, False)
        base["cleaned_match_" + field] = r.get("cleaned_match_" + field, False)
    eval_rows.append(base)

eval_df = pd.DataFrame(eval_rows)
display(eval_df)

summary = {
    "n": len(eval_df),
    "valid_json": int(eval_df.valid_json.sum()),
    "complete_generated": int(eval_df.complete_generated.sum()),
    "complete_cleaned": int(eval_df.complete_cleaned.sum()),
    "exact_generated": int(eval_df.exact_generated.sum()),
    "exact_cleaned": int(eval_df.exact_cleaned.sum()),
}

print(json.dumps(summary, indent=2))


[1/24] adv_api_01
[2/24] adv_breath_01
[3/24] adv_car_01
[4/24] adv_commit_01
[5/24] adv_constraint_01
[6/24] adv_coupler_01
[7/24] adv_coupler_02
[8/24] adv_flower_01
[9/24] adv_fold_01
[10/24] adv_house_01
[11/24] adv_idea_01
[12/24] adv_llm_01
[13/24] adv_loose_01
[14/24] adv_moore_01
[15/24] adv_observable_01
[16/24] adv_ping_01
[17/24] adv_sha_01
[18/24] adv_sha_02
[19/24] adv_socket_01
[20/24] adv_solution_01
[21/24] adv_surface_01
[22/24] adv_surface_02
[23/24] adv_tree_01
[24/24] adv_weight_01


,base_id,target_source,valid_json,complete_generated,complete_cleaned,exact_generated,exact_cleaned,generated_parse_error,expected_parse_error,generated_match_family_class,...,generated_match_boundary_conditions,cleaned_match_boundary_conditions,generated_match_preserved_function,cleaned_match_preserved_function,generated_match_failure_modes,cleaned_match_failure_modes,generated_match_witness_readout,cleaned_match_witness_readout,generated_match_residue,cleaned_match_residue
0,adv_api_01,patched,True,True,True,False,True,None,None,True,...,True,True,True,True,True,True,True,True,True,True
1,adv_breath_01,patched,True,True,True,True,True,None,None,True,...,True,True,True,True,True,True,True,True,True,True
2,adv_car_01,patched,True,True,True,True,True,None,None,True,...,True,True,True,True,True,True,True,True,True,True
3,adv_commit_01,patched,True,True,True,True,True,None,None,True,...,True,True,True,True,True,True,True,True,True,True
4,adv_constraint_01,patched,True,True,True,True,True,None,None,True,...,True,True,True,True,True,True,True,True,True,True
5,adv_coupler_01,patched,True,True,True,False,False,None,None,True,...,False,False,False,False,False,False,False,False,True,True
6,adv_coupler_02,patched,True,True,True,False,False,None,None,True,...,True,True,True,True,True,True,True,True,True,True
7,adv_flower_01,patched,True,True,True,True,True,None,None,True,...,True,True,True,True,True,True,True,True,True,True
8,adv_fold_01,patched,True,True,True,True,True,None,None,True,...,True,True,True,True,True,True,True,True,True,True
9,adv_house_01,patched,True,True,True,True,True,None,None,True,...,True,True,True,True,True,True,True,True,True,True


{
  "n": 24,
  "valid_json": 24,
  "complete_generated": 24,
  "complete_cleaned": 24,
  "exact_generated": 16,
  "exact_cleaned": 16
}


In [9]:
# ============================================================
# INSPECT FAILURES
# ============================================================
fail_df = eval_df[~eval_df["exact_cleaned"]].copy()
print("cleaned failures:", len(fail_df))
display(fail_df)

for _, row in fail_df.iterrows():
    case_id = row["base_id"]
    rec = next(r for r in records if r["base_id"] == case_id)

    print("=" * 100)
    print("CASE:", case_id)
    print()
    print("GENERATED:")
    print(json.dumps(rec["generated_contract"], indent=2, ensure_ascii=False))
    print()
    print("CLEANED:")
    print(json.dumps(rec["cleaned_contract"], indent=2, ensure_ascii=False))
    print()
    print("EXPECTED:")
    print(json.dumps(rec["expected_contract"], indent=2, ensure_ascii=False))


cleaned failures: 8


,base_id,target_source,valid_json,complete_generated,complete_cleaned,exact_generated,exact_cleaned,generated_parse_error,expected_parse_error,generated_match_family_class,...,generated_match_boundary_conditions,cleaned_match_boundary_conditions,generated_match_preserved_function,cleaned_match_preserved_function,generated_match_failure_modes,cleaned_match_failure_modes,generated_match_witness_readout,cleaned_match_witness_readout,generated_match_residue,cleaned_match_residue
5,adv_coupler_01,patched,True,True,True,False,False,None,None,True,...,False,False,False,False,False,False,False,False,True,True
6,adv_coupler_02,patched,True,True,True,False,False,None,None,True,...,True,True,True,True,True,True,True,True,True,True
13,adv_moore_01,patched,True,True,True,True,False,None,None,True,...,True,True,True,True,True,True,True,True,True,True
15,adv_ping_01,patched,True,True,True,False,False,None,None,False,...,False,False,False,False,False,False,False,False,True,True
18,adv_socket_01,patched,True,True,True,False,False,None,None,False,...,False,False,True,True,False,False,False,False,True,True
20,adv_surface_01,patched,True,True,True,False,False,None,None,True,...,True,True,True,True,True,True,True,True,True,True
21,adv_surface_02,patched,True,True,True,False,False,None,None,True,...,True,True,True,True,True,True,True,True,True,True
23,adv_weight_01,patched,True,True,True,False,False,None,None,False,...,False,False,False,False,False,False,False,False,True,True


CASE: adv_coupler_01

GENERATED:
{
  "family_class": "spinning",
  "domain_carrier": [
    "rubber",
    "coupler",
    "vacuum",
    "pump",
    "shaft"
  ],
  "forbidden_neighbor_carrier": [
    "loose",
    "centered",
    "transmit"
  ],
  "boundary_conditions": [
    "{'preserve': 'radial compression', 'reject': 'any other type of compression'}",
    "'coupler' must remain centered relative to the shaft.",
    "Rotation transmission is guaranteed as long as radial compression is applied."
  ],
  "preserved_function": "maintaining rotational motion",
  "failure_modes": [
    "general deformation of rubber",
    "surface label mismatch in collapse",
    "functionality failure due to wrong neighboring domain carrier"
  ],
  "witness_readout": "the coupler maintains contact with the shaft and rotates freely under radial compression",
  "residue": null
}

CLEANED:
{
  "family_class": "spinning",
  "domain_carrier": [
    "rubber",
    "coupler",
    "vacuum",
    "pump",
    "shaft"
  

In [10]:
# ============================================================
# SAVE EVAL OUTPUTS
# ============================================================
def json_safe_rec(r):
    return {
        "base_id": r["base_id"],
        "target_source": r["target_source"],
        "valid_json": r["valid_json"],
        "complete_generated": r["complete_generated"],
        "complete_cleaned": r["complete_cleaned"],
        "exact_generated": r["exact_generated"],
        "exact_cleaned": r["exact_cleaned"],
        "generated_parse_error": r["generated_parse_error"],
        "expected_parse_error": r["expected_parse_error"],
        "raw_generated": r["raw_generated"],
        "generated_contract": r["generated_contract"],
        "cleaned_contract": r["cleaned_contract"],
        "expected_contract": r["expected_contract"],
    }

eval_df.to_csv(OUTPUT_DIR / "slot_builder_eval_summary.csv", index=False)

with (OUTPUT_DIR / "slot_builder_eval_records.jsonl").open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(json_safe_rec(r), ensure_ascii=False) + chr(10))

manifest = {
    "model_name": MODEL_NAME,
    "adapter_dir": str(ADAPTER_DIR),
    "sft_file": str(SFT_FILE),
    "output_dir": str(OUTPUT_DIR),
    "apply_cleanup": APPLY_CLEANUP,
    "summary": summary,
}

(OUTPUT_DIR / "slot_builder_eval_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("Saved:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)


Saved:
 - slot_builder_eval_manifest.json
 - slot_builder_eval_records.jsonl
 - slot_builder_eval_summary.csv


## Ψ Readout

Use these files:

```text
slot_builder_lora_v1_eval/
  slot_builder_eval_summary.csv
  slot_builder_eval_records.jsonl
  slot_builder_eval_manifest.json
```

Interpretation:

- `valid_json`: the adapter emitted parseable JSON.
- `complete_cleaned`: all contract fields are present after cleanup.
- `exact_cleaned`: cleaned generated contract exactly matches the patched target.

Known tear from smoke test:

```text
adv_api_01 = carrier-boundary leak
```

The cleanup gate removes:

```text
persistence, retries, errors
```

from `domain_carrier` while keeping them in `forbidden_neighbor_carrier`.

Current fold:

$$
\Omega_{\text{carrier leak}}
\rightarrow
\Delta C_{\text{cleanup}}
\rightarrow
C_{\text{clean}}
$$
